# Superadditivity — Free-GPU Runner (Google Colab)

Runs the D-SGD superadditivity pipeline on a **free Colab T4**, persisting all results to **Google Drive** so nothing is lost when the session disconnects.

**How session management works here:**
- `outputs/` is symlinked into your Drive, so every finished run survives a disconnect.
- The sweep is launched with `--resume`, which skips any run that already wrote `summary.json`.
- If Colab kicks you out, just reopen this notebook and **re-run every cell top to bottom** — finished runs are skipped and the sweep continues where it stopped.

**Before burning GPU hours:** run the smoke test, then the *quick* (4-run) sweep to confirm the interaction term looks right, then the full sweep.

Set the runtime to GPU first: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# 0. Confirm a GPU is attached
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU, then reconnect.'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)

In [ ]:
# 1. Mount Google Drive (persistent storage across sessions)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone the repo and check out the working branch
REPO_URL = 'https://github.com/angelafoua/quantify_superadditivity'
BRANCH   = 'main'   # <- change to your working branch if results live elsewhere

import os
if not os.path.isdir('/content/quantify_superadditivity'):
    !git clone $REPO_URL /content/quantify_superadditivity
%cd /content/quantify_superadditivity
!git fetch origin $BRANCH && git checkout $BRANCH && git pull origin $BRANCH

In [ ]:
# 3. Install dependencies (Colab already ships a CUDA torch, so we do NOT reinstall it)
!pip install -q hydra-core==1.3.2 omegaconf==2.3.0 h5py networkx statsmodels seaborn tabulate tqdm scikit-learn pandas scipy
!pip install -q -e . --no-deps
!python scripts/validate_setup.py

In [ ]:
# 4. Point outputs/ at a Drive folder so results persist across disconnects
DRIVE_OUT = '/content/drive/MyDrive/superadditivity_outputs'
import os
os.makedirs(DRIVE_OUT, exist_ok=True)
if os.path.islink('outputs') or os.path.exists('outputs'):
    !rm -rf outputs
os.symlink(DRIVE_OUT, 'outputs')
print('outputs ->', os.path.realpath('outputs'))

In [ ]:
# 5. Smoke test (~1-2 min): proves the full pipeline runs before spending real GPU hours
!python scripts/run_experiment.py data=iid graph=erdos_renyi training.total_rounds=5 experiment_name=smoke_test

In [ ]:
# 6a. QUICK validation sweep: 4 runs (1 seed per cell). Confirms the interaction
#     effect is directionally present. Resumable.
!python scripts/run_sweep.py --experiment core_factorial_quick --resume

In [ ]:
# 6b. FULL core factorial: 60 runs. Spans several Colab sessions. --resume makes
#     it safe to re-run this cell after any disconnect; finished runs are skipped.
!python scripts/run_sweep.py --experiment core_factorial --resume

In [ ]:
# 7. Analysis + figures (run once the sweep above has finished)
!python scripts/run_analysis.py --results_dir outputs/core_factorial_quick
!python scripts/run_visualization.py --results_dir outputs/core_factorial_quick

## Running on Kaggle instead (30 GPU-hrs/week, 2× T4)

Kaggle has no Drive mount; persist results by committing the notebook (**Save Version**), which snapshots `/kaggle/working`. To carry results between sessions, add your previous output as an input dataset. Cells:

```python
!git clone https://github.com/angelafoua/quantify_superadditivity /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q hydra-core==1.3.2 omegaconf==2.3.0 h5py networkx statsmodels seaborn tabulate tqdm && pip install -q -e . --no-deps
# persist outputs under /kaggle/working so Save Version keeps them
import os; os.makedirs('/kaggle/working/outputs', exist_ok=True)
if os.path.exists('outputs'): os.remove('outputs') if os.path.islink('outputs') else None
os.symlink('/kaggle/working/outputs', 'outputs')
!python scripts/run_sweep.py --experiment core_factorial --resume
```

## If runs are too slow / too many for the free tier

- Use `--experiment core_factorial_quick` (4 runs) for the headline result.
- Lower `training.total_rounds` (e.g. 300) in `configs/training/default.yaml`.
- Swap `model: resnet18` for the lighter `convnet` in the experiment config.
- If you hit CUDA OOM, drop `batch_size` to 32 in `configs/training/default.yaml`.